# Hypothesis Testing: Risk Differences Across Segments
This notebook performs statistical hypothesis tests to evaluate:
- Claim frequency differences (proportion tests)
- Claim severity differences (mean tests on positive claims)
- Margin differences 
across groups like provinces, postal codes, and gender.


In [10]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_parquet("../data/processed/df_clean.parquet")
# Ensure HasClaim column exists
if 'HasClaim' not in df.columns:
    df['HasClaim'] = df['TotalClaims'] > 0
print("Data shape:", df.shape)
# Optionally preview:
df[['Province','PostalCode','Gender','HasClaim','TotalClaims','Margin']].head()


Data shape: (1000098, 56)


,Province,PostalCode,Gender,HasClaim,TotalClaims,Margin
0,Gauteng,1459,Not specified,False,0.0,21.929825
1,Gauteng,1459,Not specified,False,0.0,21.929825
2,Gauteng,1459,Not specified,False,0.0,0.000000
3,Gauteng,1459,Not specified,False,0.0,512.848070
4,Gauteng,1459,Not specified,False,0.0,0.000000


we load the cleaned DataFrame, ensure the flag HasClaim, and inspect relevant columns.

In [11]:
test_results = pd.read_csv("../results/stat_tests/stat_tests_results.csv")
print("Test results:")
display(test_results)


Test results:


,test,group_col,groupA,groupB,nA,nB,metricA,metricB,statistic,p_value,reject_null
0,frequency,Province,Gauteng,Western Cape,393865,170796,0.003356,0.002166,7.515654,5.662137e-14,True
1,mean_TotalClaims,Province,Gauteng,Western Cape,1322,370,22243.878396,28095.849881,-2.168535,3.059896e-02,True
2,frequency,PostalCode,2000,122,133498,49171,0.003641,0.004271,-1.939401,5.245248e-02,False
3,mean_TotalClaims,PostalCode,2000,122,486,210,19196.413727,18162.025865,0.385376,7.002080e-01,False
4,mean_Margin,PostalCode,2000,122,133498,49171,-8.111944,-22.859806,1.163915,2.444624e-01,False
5,frequency,Gender,Male,Female,42817,6755,0.002195,0.002073,0.201261,8.404941e-01,False
6,mean_TotalClaims,Gender,Male,Female,94,14,14858.552294,17874.721303,-0.579020,5.680287e-01,False


In [12]:
for idx, row in test_results.iterrows():
    group_col = row['group_col']
    A, B = row['groupA'], row['groupB']
    test_type = row['test']  # e.g., 'frequency' or 'mean_TotalClaims'
    stat = row['statistic']
    p = row['p_value']
    decision = "Reject H0" if row['reject_null'] else "Fail to reject H0"
    # Compute basic description:
    if test_type == 'frequency':
        metric = "claim frequency"
    elif test_type.startswith('mean_'):
        metric = "mean " + test_type.split('_',1)[1]
    else:
        metric = test_type
    print(f"- {group_col}: {A} vs {B}, {metric}, statistic={stat:.4f}, p={p:.4e} → {decision}")


- Province: Gauteng vs Western Cape, claim frequency, statistic=7.5157, p=5.6621e-14 → Reject H0
- Province: Gauteng vs Western Cape, mean TotalClaims, statistic=-2.1685, p=3.0599e-02 → Reject H0
- PostalCode: 2000 vs 122, claim frequency, statistic=-1.9394, p=5.2452e-02 → Fail to reject H0
- PostalCode: 2000 vs 122, mean TotalClaims, statistic=0.3854, p=7.0021e-01 → Fail to reject H0
- PostalCode: 2000 vs 122, mean Margin, statistic=1.1639, p=2.4446e-01 → Fail to reject H0
- Gender: Male vs Female, claim frequency, statistic=0.2013, p=8.4049e-01 → Fail to reject H0
- Gender: Male vs Female, mean TotalClaims, statistic=-0.5790, p=5.6803e-01 → Fail to reject H0



# Interpretation of initial tests


# Province
 “Gauteng vs Western Cape, claim frequency difference is significant (p≈5.66e-14), but effect size is only ~0.12% higher in Gauteng—practically a small difference though statistically significant due to large N. Business implication: a very slight premium uplift might be considered for Gauteng, but effect is minimal.”

# Severity
 “Gauteng vs Western Cape, mean TotalClaims difference is significant (p≈0.03), but Cohen’s d ≈ –0.15 (small effect) and Mann-Whitney U p≈0.36 (no significant difference non-parametrically). This suggests severity distributions are skewed; t-test result might not be reliable. Business implication: do not adjust severity-based pricing solely on this difference without further analysis.”

# PostalCode
 “2000 vs 122: frequency p≈0.052 fail to reject; severity p≈0.70 fail; margin p≈0.24 fail. No evidence of difference.”

# Gender
 “Male vs Female: frequencies and severity both non-significant, and female sample size for severity is small (<30), so unreliable. Do not segment by gender here.”

In [13]:
summary = []
for _, row in test_results.iterrows():
    group_col = row['group_col']
    A, B = row['groupA'], row['groupB']
    metric = ("frequency" if row['test']=='frequency'
              else "severity" if "TotalClaims" in row['test']
              else "margin" if "Margin" in row['test']
              else row['test'])
    effect = None  # to be computed next
    summary.append({
        'Segment': group_col,
        'Group A': A,
        'Group B': B,
        'Metric': metric,
        'Statistic': row['statistic'],
        'p-value': row['p_value'],
        'Decision': "Reject" if row['reject_null'] else "Fail to reject"
    })
summary_df = pd.DataFrame(summary)
display(summary_df)



,Segment,Group A,Group B,Metric,Statistic,p-value,Decision
0,Province,Gauteng,Western Cape,frequency,7.515654,5.662137e-14,Reject
1,Province,Gauteng,Western Cape,severity,-2.168535,3.059896e-02,Reject
2,PostalCode,2000,122,frequency,-1.939401,5.245248e-02,Fail to reject
3,PostalCode,2000,122,severity,0.385376,7.002080e-01,Fail to reject
4,PostalCode,2000,122,margin,1.163915,2.444624e-01,Fail to reject
5,Gender,Male,Female,frequency,0.201261,8.404941e-01,Fail to reject
6,Gender,Male,Female,severity,-0.579020,5.680287e-01,Fail to reject


In [14]:
# Effect size for proportions: difference pA - pB
def prop_diff(df, group_col, value_col, A, B):
    dfA = df[df[group_col] == A]
    dfB = df[df[group_col] == B]
    pA = dfA[value_col].mean()
    pB = dfB[value_col].mean()
    return pA - pB

# Cohen’s d for two samples
def cohen_d(seriesA, seriesB):
    a = seriesA.dropna()
    b = seriesB.dropna()
    n1, n2 = len(a), len(b)
    # Pooled standard deviation
    s1, s2 = a.std(ddof=1), b.std(ddof=1)
    pooled_sd = np.sqrt(((n1-1)*s1*s1 + (n2-1)*s2*s2) / (n1 + n2 - 2))
    if pooled_sd == 0:
        return np.nan
    return (a.mean() - b.mean()) / pooled_sd


In [15]:
effects = []
for _, row in test_results.iterrows():
    col = row['group_col']
    A, B = row['groupA'], row['groupB']
    if row['test'] == 'frequency':
        diff = prop_diff(df, col, 'HasClaim', A, B)
        effects.append(diff)
    elif row['test'] == 'mean_TotalClaims':
        # severity: consider only positive claims
        seriesA = df[(df[col] == A) & (df['HasClaim'])]['TotalClaims']
        seriesB = df[(df[col] == B) & (df['HasClaim'])]['TotalClaims']
        d = cohen_d(seriesA, seriesB)
        effects.append(d)
    elif row['test'] == 'mean_Margin':
        # margin across all
        seriesA = df[df[col] == A]['Margin']
        seriesB = df[df[col] == B]['Margin']
        d = cohen_d(seriesA, seriesB)
        effects.append(d)
    else:
        effects.append(np.nan)
test_results['EffectSize'] = effects
display(test_results[['group_col','groupA','groupB','test','p_value','reject_null','EffectSize']])


,group_col,groupA,groupB,test,p_value,reject_null,EffectSize
0,Province,Gauteng,Western Cape,frequency,5.662137e-14,True,0.001190
1,Province,Gauteng,Western Cape,mean_TotalClaims,3.059896e-02,True,-0.149985
2,PostalCode,2000,122,frequency,5.245248e-02,False,-0.000630
3,PostalCode,2000,122,mean_TotalClaims,7.002080e-01,False,0.034799
4,PostalCode,2000,122,mean_Margin,2.444624e-01,False,0.006822
5,Gender,Male,Female,frequency,8.404941e-01,False,0.000123
6,Gender,Male,Female,mean_TotalClaims,5.680287e-01,False,-0.120051


# Frequency difference for Province
0.00119 means 0.119% absolute difference in claim probability. With overall claim rate ~0.28%, this is a relative increase of ~40%, but absolute change small. Severity Cohen’s d ≈ –0.15 is a small effect. Postal code differences and gender effect sizes are near zero (<0.01), negligible.

We compute difference in proportions for frequency tests; Cohen’s d for mean tests. Values: small (<0.2), medium (~0.5), large (>0.8) interpretations.

In [16]:
from scipy.stats import mannwhitneyu
# Example between Gauteng and Western Cape severity:
a = df[(df['Province']=='Gauteng')&(df['HasClaim'])]['TotalClaims']
b = df[(df['Province']=='Western Cape')&(df['HasClaim'])]['TotalClaims']
stat, p_mw = mannwhitneyu(a, b, alternative='two-sided')
print("Mann-Whitney U:", stat, "p-value:", p_mw)


Mann-Whitney U: 236926.5 p-value: 0.35695564497624155


#### Robustness Check for Severity (Province)
We perform Mann-Whitney U test because TotalClaims is highly skewed.
- Gauteng vs Western Cape: U-statistic = 236926.5, p-value ≈ 0.357 → fail to reject null.
- This contradicts the t-test result (p≈0.03). Because severity distribution is zero-inflated and heavy-tailed, the non-parametric result is more trustworthy here. Therefore, we conclude no strong evidence of difference in claim severity between provinces.
